# Experimental test 1 Result

* vllm기준으로 accuracy와 f1 score를 테스트
* few-shot테스트를 위해서 sc.Self_Consistency의 두번째 파라미터를 1, 2, 3, 4 로 변경하여 실험 수행
* 이외의 파라미터는 고정 
    * fewshot 테스트에 활용할 질문의 개수  : 60
    * 소스코드 포함여부  : 'Y'           
    * 반복횟수 : 5회                
    * 시스템프롬프트 'sys_prompt10'
    * self-consistency 횟수 : 5
    * temperature : 0.01
    * 엑셀버전 : 'ver7'
* 이후 결과에 대해서 스코어 비교 진행 


In [1]:
import sys, os
import re
import numpy as np
from sklearn import metrics
import pandas as pd



In [2]:
# /mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/experiment/run_id_1/sc_vq_result_4_30_Y_30_sys_prompt10_5_0.01_ver7_0.csv
def sc_calc_acc_condition_with_temp_with_sc(run_id, llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    acc_list = []
    path = f'/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/validation/{run_id}'
    file_list = os.listdir(path)
    print(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')]

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['result'] = tmp['result_long'].apply(lambda x: int(m.group(1)) if (m := re.search(r'<Difficulty Level>\s*(\d)\s*', x)) else None)
            tmp['answer'] = tmp['answer'].apply(lambda x: int(m.group(1)) if (m := re.search(r'<Difficulty Level>\s*(\d)\s*', x)) else None)
            # tmp['answer'] = tmp['answer'].map({'Difficulty Level : Intermediate' : int(1), 'Difficulty Level : Advanced' : int(2), 'Difficulty Level : Basic' : int(0)})

            
            eval_df = tmp.groupby(['id', 'answer', 'result']).count()['question'].reset_index().rename(columns={'question' : 'count'})
            # eval_df = eval_df.sort_values(by = ['id', 'count'], ascending=[True, False]).groupby(['id', 'answer']).head(1)
            eval_df = eval_df[eval_df['count'] == sc_num]
            eval_df.loc[:, 'equal_yn'] = np.where(eval_df['answer']==eval_df['result'], 1, 0)
            acc = (eval_df['equal_yn'].sum()/eval_df.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, eval_df], axis =0)
            
        df.loc[:, 'equal_yn'] = np.where(df['answer']==df['result'], 1, 0)
        y_true = df['result']
        y_pred = df['answer']
        print(f"value count for golen : {df['answer'].value_counts()}")
        print(f"value count for o_result : {df['result'].value_counts()}")
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list, df


In [7]:
# sc_vq_result_4_50_Y_50_sys_prompt10_5_0.01_ver8_9.csv

list_, df_ =         sc_calc_acc_condition_with_temp_with_sc('run_id_10', 'vq', 4, 50, 'Y', 50, 'sys_prompt10', 5,  0.01, 'ver8')
print(list_)

sc_vq_result_4_50_Y_50_sys_prompt10_5_0.01_ver8
value count for golen : answer
1    1565
2    1088
0    1033
Name: count, dtype: int64
value count for o_result : result
1    1378
2    1250
0    1058
Name: count, dtype: int64
              precision    recall  f1-score   support

           0      0.991     0.968     0.979      1058
           1      0.847     0.962     0.900      1378
           2      0.960     0.835     0.893      1250

    accuracy                          0.921      3686
   macro avg      0.932     0.922     0.924      3686
weighted avg      0.926     0.921     0.921      3686

vq_result_4_50_Y :  92.05100379815518
[np.float64(100.0), np.float64(90.9090909090909), np.float64(88.57142857142857), np.float64(94.44444444444444), np.float64(97.2972972972973), np.float64(92.5), np.float64(94.5945945945946), np.float64(97.61904761904762), np.float64(97.5), np.float64(90.32258064516128), np.float64(92.3076923076923), np.float64(91.42857142857143), np.float64(93.54838709677

In [10]:
# sc_vq_result_4_50_Y_50_sys_prompt10_5_0.01_ver8_9.csv

list_, df_ =         sc_calc_acc_condition_with_temp_with_sc('run_id_112', 'vq', 4, 30, 'Y', 30, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

sc_vq_result_4_30_Y_30_sys_prompt10_5_0.01_ver7
value count for golen : answer
1    138
0     30
2     29
Name: count, dtype: int64
value count for o_result : result
1    103
2     67
0     27
Name: count, dtype: int64
              precision    recall  f1-score   support

           0      0.533     0.593     0.561        27
           1      0.645     0.864     0.739       103
           2      1.000     0.433     0.604        67

    accuracy                          0.680       197
   macro avg      0.726     0.630     0.635       197
weighted avg      0.750     0.680     0.669       197

vq_result_4_30_Y :  68.02030456852792
[np.float64(54.54545454545454), np.float64(80.95238095238095), np.float64(52.38095238095239), np.float64(68.18181818181817), np.float64(66.66666666666666), np.float64(72.22222222222221), np.float64(75.0), np.float64(68.18181818181817), np.float64(61.111111111111114), np.float64(84.21052631578947)]


In [ ]:
golden = pd.read_csv('/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/q_output_code_y_105.csv')

In [11]:
tmp = pd.read_csv('/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/validation/run_id_112/sc_vq_result_4_30_Y_30_sys_prompt10_5_0.01_ver7_0.csv', index_col =0)

In [14]:
tmp.head(25)
# .apply(lambda x: int(m.group(1)) if (m := re.search(r'<Difficulty Level>\s*(\d)\s*', x)) else None)

,id,question,answer,result_long
0,70811411,<Title>How to loop through rows of specific co...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1
1,70811411,<Title>How to loop through rows of specific co...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1
2,70811411,<Title>How to loop through rows of specific co...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1
3,70811411,<Title>How to loop through rows of specific co...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1
4,70811411,<Title>How to loop through rows of specific co...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1
5,73784250,<Title>Can I call an instance of a class using...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1
6,73784250,<Title>Can I call an instance of a class using...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1
7,73784250,<Title>Can I call an instance of a class using...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1
8,73784250,<Title>Can I call an instance of a class using...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1
9,73784250,<Title>Can I call an instance of a class using...,<Difficulty Level>1</Difficulty Level>,<Difficulty Level>1


In [15]:
tmp['result'] = tmp['result_long'].apply(lambda x: int(m.group(1)) if (m := re.search(r'<Difficulty Level>\s*(\d)\s*', x)) else None)
tmp['answer'] = tmp['answer'].apply(lambda x: int(m.group(1)) if (m := re.search(r'<Difficulty Level>\s*(\d)\s*', x)) else None)
# tmp['answer'] = tmp['answer'].map({'Difficulty Level : Intermediate' : int(1), 'Difficulty Level : Advanced' : int(2), 'Difficulty Level : Basic' : int(0)})


In [16]:
tmp.groupby(['id', 'answer', 'result']).count()['question'].reset_index().rename(columns={'question' : 'count'})

,id,answer,result,count
0,70811411,1,1,5
1,70861843,1,1,5
2,70870485,0,1,5
3,71230785,1,0,5
4,71407368,1,1,3
5,71407368,1,2,2
6,72898073,1,0,5
7,73784250,1,1,5
8,73981914,1,2,5
9,73983391,0,0,1


In [19]:
eval_df = tmp.groupby(['id', 'answer', 'result']).count()['question'].reset_index().rename(columns={'question' : 'count'})
# eval_df = eval_df.sort_values(by = ['id', 'count'], ascending=[True, False]).groupby(['id', 'answer']).head(1)
eval_df = eval_df[eval_df['count'] == 5]

In [22]:
eval_df.loc[:, 'equal_yn'] = np.where(eval_df['answer']==eval_df['result'], 1, 0)
acc = (eval_df['equal_yn'].sum()/eval_df.shape[0])*100  
print(acc)

68.18181818181817


In [23]:
eval_df

,id,answer,result,count,equal_yn
0,70811411,1,1,5,1
1,70861843,1,1,5,1
2,70870485,0,1,5,0
3,71230785,1,0,5,0
6,72898073,1,0,5,0
7,73784250,1,1,5,1
8,73981914,1,2,5,0
11,74262026,0,1,5,0
12,74468667,0,1,5,0
15,75153050,1,1,5,1


In [18]:
tmp.loc[tmp['id'] == 78046944, 'question'].values[0]

'<Title>Difficulty Using gettext_lazy in Django 4 Settings with django-stubs, Resulting in Import Cycle and mypy Type Inference Error</Title>. <Question>Problem:\nMypy Error:\nThe mypy error encountered is as follows:\nerror:Import cycle from Django settings module prevents type inference for \'LANGUAGES\' [misc]\nI\'m encountering challenges with the usage of gettext_lazy in Django 4 settings while utilizing django-stubs. The recent update in Django 4 brought changes to the typing of gettext_lazy, where the old return type was str, and the new type is _StrPromise. The issue arises from the fact that _StrPromise is defined in "django-stubs/utils/functional.pyi,"; and within this file, there is also an import of django.db.model which imports settings. This creates a circular import.\ncurrent module-version:\ntyping\n```python\nmypy = "1.7";\ndjango-stubs = "4.2.7";\n```\nDjango dependencies\nDjango = "4.2.10";\nSeeking advice on a cleaner and more sustainable solution to the circular im

In [ ]:
tmp.groupby(['id', 'answer', 'result']).count()['question'].reset_index().rename(columns={'question' : 'count'})

In [ ]:
end select_fewshot_for_e : dict_keys([np.int64(76393832), np.int64(73983391), np.int64(73448504), np.int64(77027214), np.int64(70507747), np.int64(72935086), np.int64(76618653), np.int64(70264812), np.int64(78971762), np.int64(75582323), np.int64(74672060), np.int64(71497553), np.int64(70627235), np.int64(74537755), np.int64(70895374), np.int64(72645388), np.int64(74316658), np.int64(75178436), np.int64(76187123), np.int64(74514741), np.int64(70217014), np.int64(70738995), np.int64(74933076), np.int64(78812288), np.int64(71824874), np.int64(75852349), np.int64(78572172), np.int64(79092651), np.int64(78042738), np.int64(76911199)])

In [ ]:
gold = pd.read_csv('/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/q_output_code_y_snapshot2_md.csv')

In [ ]:
vali = pd.read_csv('/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/q_output_code_y_snapshot2_validation.csv')

In [ ]:
vali[vali['id'] == 76393832]

In [ ]:
gold = pd.read_csv('/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/q_output_code_y_snapshot2_md.csv')